In [0]:
%sql

use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.functions import lit

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202510")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# debug
display(par_month)

In [0]:
if par_month in ['202601','202602','202603']:
    date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/date_chula.csv'
else:
    date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/day_type_apr_june_26.csv'

prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_footfall.parquet'
profile_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/profile_chula.csv'

In [0]:
df_profile = spark.read.option("header", "true").option("inferSchema", "true")\
    .csv(profile_path)\
    .select("name","lat", "lon",'province','district','sub_district','Developer split_1','Developer split_2')
df_profile = df_profile.withColumnRenamed('lat','latitude').withColumnRenamed('lon','longitude')

In [0]:
df = spark.read.parquet(prep_path)\
    .withColumn('par_month', lit(par_month))
        # .filter(F.col('msisdn')=='739592555467bb368e7a9348fb70b5b2e546944c067e0db259d9ca73579945bc')
print('total :',df.select('msisdn').distinct().count())
print('bmr :',df.filter(F.col('is_bmr')==1).select('msisdn').distinct().count())
print('non bmr :',df.filter(F.col('is_non_bmr')==1).select('msisdn').distinct().count())
print('foriegner :',df.filter(F.col('is_foriegner')==1).select('msisdn').distinct().count())
print('unknown :',df.filter(((F.col('is_bmr')==0)&(F.col('is_non_bmr')==0)&(F.col('is_foriegner')==0))).select('msisdn').distinct().count()) # join 360 inner กรองออก

# total : 2478678
# bmr : 1722020
# non bmr : 253959
# foriegner : 364219
# unknown : 139924

In [0]:
df_intermediate = df.join(F.broadcast(df_profile), 'name', 'left')

In [0]:
# 1-year dataset
df_date = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(date_path)).select('date','WEEK','day_type_final')

# define month
df_month = df_date.filter(df_date.date >= int(str(par_month)+'01'))
df_month = df_month.filter(df_month.date <= int(str(par_month)+'31'))

# define full week (7-day within the month)
df_week_in_month = df_month.groupBy("WEEK").agg(F.count('date').alias('N')).filter(F.col('N') == 7)
n_week = df_week_in_month.count()
min_week = df_week_in_month.select(F.min('WEEK')).collect()[0][0]
max_week = df_week_in_month.select(F.max('WEEK')).collect()[0][0]

# debug
print('n_week = ' + str(n_week))
print('min_week = ' + str(min_week))
print('max_week = ' + str(max_week))

In [0]:
# left เอา number_week , day_type_final
df_intermediate_date = df_intermediate.join(F.broadcast(df_date), df_intermediate.par_day == df_date.date, 'left')

### Visit Frequency (Day)	
Visit frequency is categorized based on the number of days a customer visits the mall within a month.
- 1 
- 2–3 
- 4–10 
- 11–20 
- 20–30 (21-31)

In [0]:
df_visit_freq = df_intermediate_date.groupBy('msisdn','name','par_month').agg(F.countDistinct("par_day").alias('visit_freq_num'))

# Visit Frequency (Day)
df_visit_freq = df_visit_freq.withColumn('visit_frequency_(day)', 
                                         F.when(F.col('visit_freq_num')==1, F.lit('1'))
                                          .when(F.col('visit_freq_num').between(2,3), F.lit('2–3'))
                                          .when(F.col('visit_freq_num').between(4,10), F.lit('4–10'))
                                          .when(F.col('visit_freq_num').between(11,20), F.lit('11–20'))
                                          .when(F.col('visit_freq_num').between(21,31), F.lit('21–31'))
)

df_intermediate_freq = df_intermediate_date.join(df_visit_freq.select('msisdn','name','par_month','visit_frequency_(day)','visit_freq_num'), on=['msisdn','name','par_month'], how='left')

# debug
# display(df_intermediate_freq.limit(5))

### Weekly Mall Visitors	
Customers who visit the mall on a regular weekly basis, defined as individuals who are present at the mall at least once per week.

In [0]:
df_visit_weekly = df_intermediate_freq.filter(F.col('WEEK').between(min_week, max_week)).groupBy('msisdn','name').agg(F.countDistinct("WEEK").alias('N'))

df_visit_weekly = df_visit_weekly.withColumn('weekly_mall_visitors', F.when(F.col('N')==n_week, F.lit(1))
                                               .otherwise(F.lit(0))
)

df_intermediate_weekly = df_intermediate_freq.join(df_visit_weekly.select('msisdn','name','weekly_mall_visitors'), on=['msisdn','name'], how='left')

# display(df_intermediate_weekly.limit(5))

In [0]:
# lowercase
df_intermediate_placetype = df_intermediate_weekly.withColumn('province', F.lower(F.col('province')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('district', F.lower(F.col('district')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('sub_district', F.lower(F.col('sub_district')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('home_province', F.lower(F.col('geog_resident_location_v1_province_en_cat')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('home_district', F.lower(F.col('geog_resident_location_v1_district_en_cat')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('home_subdistrict', F.lower(F.col('geog_resident_location_v1_sub_district_en_cat')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('work_province', F.lower(F.col('geog_work_location_v1_province_en_cat')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('work_district', F.lower(F.col('geog_work_location_v1_district_en_cat')))
df_intermediate_placetype = df_intermediate_placetype.withColumn('work_subdistrict', F.lower(F.col('geog_work_location_v1_sub_district_en_cat')))

# conditions
df_intermediate_placetype = df_intermediate_placetype.withColumn('place_type', 
                        F.when(((F.col('home_province')==F.col('province')) 
                                & (F.col('home_district')==F.col('district'))
                                & (F.col('home_subdistrict')==F.col('sub_district'))
                                ) &
                                ((F.col('work_province')!=F.col('province')) 
                                 | (F.col('work_district')!=F.col('district')) 
                                 | (F.col('work_subdistrict')!=F.col('sub_district')) 
                                 | (F.col('work_subdistrict').isNull())
                                 )
                                , F.lit('home'))
                        .when(((F.col('work_province')==F.col('province')) 
                               & (F.col('work_district')==F.col('district'))
                               & (F.col('work_subdistrict')==F.col('sub_district'))
                               ) &
                                ((F.col('home_province')!=F.col('province')) 
                                 | (F.col('home_district')!=F.col('district')) 
                                 | (F.col('home_subdistrict')!=F.col('sub_district')) 
                                 | (F.col('home_subdistrict').isNull())
                                 )
                                , F.lit('work'))
                        .when(((F.col('home_province')==F.col('province'))
                               & (F.col('home_district')==F.col('district'))
                               & (F.col('home_subdistrict')==F.col('sub_district'))) 
                              &
                                ((F.col('work_province')==F.col('province')) 
                                 & (F.col('work_district')==F.col('district'))
                                 & (F.col('work_subdistrict')==F.col('sub_district')))
                                , F.lit('home_work'))
                        .when((((F.col('home_province')!=F.col('province')) 
                                | (F.col('home_district')!=F.col('district'))
                                | (F.col('home_subdistrict')!=F.col('sub_district'))
                                ) | (F.col('home_subdistrict').isNull())) 
                              &
                                (((F.col('work_province')!=F.col('province')) 
                                  | (F.col('work_district')!=F.col('district'))
                                  | (F.col('work_subdistrict')!=F.col('sub_district'))
                                  ) | (F.col('work_subdistrict').isNull())) 
                                &
                                ((F.col('home_subdistrict').isNotNull()) | (F.col('work_subdistrict').isNotNull()))
                                , F.lit('thirdplace'))
                        .when((F.col('home_province').isNull()) 
                              & (F.col('home_district').isNull()) 
                              & (F.col('home_subdistrict').isNull()) 
                              & (F.col('work_province').isNull()) 
                              & (F.col('work_district').isNull())
                              & (F.col('work_subdistrict').isNull())
                                , F.lit('home_work_unidentified'))
                        .otherwise(F.lit('other')) # debug
)

# debug: expect return 0 row
# display(df_intermediate_placetype.select('msisdn','place_type','work_province','work_subdistrict','home_province','home_subdistrict','province','sub_district').filter(F.col('place_type')=='other').limit(20))

In [0]:
# count number of visit per group
# df_group_visit_ = df_intermediate_placetype.filter((F.col('place_type')=='home_work_unidentified') 
#                                                       | (F.col('place_type')=='thirdplace')
#                                                       | (F.col('place_type')=='work')
#                                                       )
df_group_visit_ = df_intermediate_placetype
df_group_visit = df_group_visit_.groupBy('msisdn','Developer split_1').agg(F.countDistinct("par_day").alias('visit_freq'))
df_group_visit_2 = df_group_visit_.groupBy('msisdn','Developer split_2').agg(F.countDistinct("par_day").alias('visit_freq'))
df_group_visit_2 = df_group_visit_2.filter(F.col('Developer split_2')=='SPW Group')

# CPN_CBD
df_group_visit = df_group_visit.withColumn('cpn_cbd_customers', 
                                           F.when((F.col('visit_freq')>=1) & (F.col('Developer split_1')=='CPN_CBD'), F.lit(1))
                                          .otherwise(F.lit(0))
)
df_group_visit = df_group_visit.withColumn('cpn_cbd_frequent_customers', 
                                           F.when((F.col('visit_freq')>=3) & (F.col('Developer split_1')=='CPN_CBD'), F.lit(1))
                                          .otherwise(F.lit(0))
) 
# CPN_Non-CBD 
df_group_visit = df_group_visit.withColumn('cpn_non_cbd_customers', 
                                           F.when((F.col('visit_freq')>=1) &  (F.col('Developer split_1')=='CPN_Non-CBD'), F.lit(1))
                                          .otherwise(F.lit(0))
)
df_group_visit = df_group_visit.withColumn('cpn_non_cbd_frequent_customers', 
                                           F.when((F.col('visit_freq')>=3) & (F.col('Developer split_1')=='CPN_Non-CBD'), F.lit(1))
                                          .otherwise(F.lit(0))
)
# SPW_ICONICS
df_group_visit = df_group_visit.withColumn('spw_iconics_customers', 
                                           F.when((F.col('visit_freq')>=1) & (F.col('Developer split_1')=='SPW_ICONICS'), F.lit(1))
                                          .otherwise(F.lit(0))
)
df_group_visit = df_group_visit.withColumn('spw_iconics_frequent_customers', 
                                           F.when((F.col('visit_freq')>=3) & (F.col('Developer split_1')=='SPW_ICONICS'), F.lit(1))
                                          .otherwise(F.lit(0))
)
# SPW_SPDSCSD
df_group_visit = df_group_visit.withColumn('spw_spdscsd_customers', 
                                           F.when((F.col('visit_freq')>=1) & (F.col('Developer split_1')=='SPW_SPDSCSD'), F.lit(1))
                                          .otherwise(F.lit(0))
)
df_group_visit = df_group_visit.withColumn('spw_spdscsd_frequent_customers', 
                                           F.when((F.col('visit_freq')>=3) & (F.col('Developer split_1')=='SPW_SPDSCSD'), F.lit(1))
                                          .otherwise(F.lit(0))
)
# TCC
df_group_visit = df_group_visit.withColumn('tcc_customers', 
                                           F.when((F.col('visit_freq')>=1) & (F.col('Developer split_1')=='TCC'), F.lit(1))
                                          .otherwise(F.lit(0))
)
df_group_visit = df_group_visit.withColumn('tcc_frequent_customers', 
                                           F.when((F.col('visit_freq')>=3) & (F.col('Developer split_1')=='TCC'), F.lit(1))
                                          .otherwise(F.lit(0))
)
# The Mall_CBD
df_group_visit = df_group_visit.withColumn('the_mall_cbd_customers', 
                                           F.when((F.col('visit_freq')>=1) & (F.col('Developer split_1')=='The Mall_CBD'), F.lit(1))
                                          .otherwise(F.lit(0))
)
df_group_visit = df_group_visit.withColumn('the_mall_cbd_frequent_customers', 
                                           F.when((F.col('visit_freq')>=3) & (F.col('Developer split_1')=='The Mall_CBD'), F.lit(1))
                                          .otherwise(F.lit(0))
)
# The Mall_Non-CBD
df_group_visit = df_group_visit.withColumn('the_mall_non_cbd_customers', 
                                    F.when((F.col('visit_freq')>=1) & (F.col('Developer split_1')=='The Mall_Non-CBD'), F.lit(1))
                                    .otherwise(F.lit(0))
)
df_group_visit = df_group_visit.withColumn('the_mall_non_cbd_frequent_customers', 
                                    F.when((F.col('visit_freq')>=3) & (F.col('Developer split_1')=='The Mall_Non-CBD'), F.lit(1))
                                    .otherwise(F.lit(0))
)
# SPW Group
df_group_visit_2 = df_group_visit_2.withColumn('spw_group_customers', 
                                    F.when((F.col('visit_freq')>=1) & (F.col('Developer split_2')=='SPW Group'), F.lit(1))
                                    .otherwise(F.lit(0))
)
df_group_visit_2 = df_group_visit_2.withColumn('spw_group_frequent_customers', 
                                    F.when((F.col('visit_freq')>=3) & (F.col('Developer split_2')=='SPW Group'), F.lit(1))
                                    .otherwise(F.lit(0))
)

# combine every group into one
df_group_visit = df_group_visit.groupBy('msisdn').agg(
  F.sum('cpn_cbd_customers').alias('cpn_cbd_customers'),
  F.sum('cpn_cbd_frequent_customers').alias('cpn_cbd_frequent_customers'),
  F.sum('cpn_non_cbd_customers').alias('cpn_non_cbd_customers'),
  F.sum('cpn_non_cbd_frequent_customers').alias('cpn_non_cbd_frequent_customers'),
  F.sum('spw_iconics_customers').alias('spw_iconics_customers'),
  F.sum('spw_iconics_frequent_customers').alias('spw_iconics_frequent_customers'),
  F.sum('spw_spdscsd_customers').alias('spw_spdscsd_customers'),
  F.sum('spw_spdscsd_frequent_customers').alias('spw_spdscsd_frequent_customers'),
  F.sum('tcc_customers').alias('tcc_customers'),
  F.sum('tcc_frequent_customers').alias('tcc_frequent_customers'),
  F.sum('the_mall_cbd_customers').alias('the_mall_cbd_customers'),
  F.sum('the_mall_cbd_frequent_customers').alias('the_mall_cbd_frequent_customers'),
  F.sum('the_mall_non_cbd_customers').alias('the_mall_non_cbd_customers'),
  F.sum('the_mall_non_cbd_frequent_customers').alias('the_mall_non_cbd_frequent_customers')
)

# join into the main df
df_intermediate_visit = df_intermediate_placetype.join(df_group_visit, 'msisdn', 'left')
df_group_visit_2 = df_group_visit_2.select('msisdn','spw_group_customers','spw_group_frequent_customers')
df_intermediate_visit = df_intermediate_visit.join(df_group_visit_2, 'msisdn', 'left')

# debug
# display(df_intermediate_visit.limit(5))

In [0]:

# count number of weekly active customers
# df_group_visit_weekly_ = df_intermediate_visit.filter((F.col('place_type')=='home_work_unidentified') 
#                                                       | (F.col('place_type')=='thirdplace')
#                                                       | (F.col('place_type')=='work')
#                                                       )
df_group_visit_weekly = df_intermediate_visit.filter(F.col('WEEK').between(min_week, max_week)).groupBy('msisdn','Developer split_1').agg(F.countDistinct("WEEK").alias('N'))
df_group_visit_weekly_2 = df_intermediate_visit.filter(F.col('WEEK').between(min_week, max_week)).groupBy('msisdn','Developer split_2').agg(F.countDistinct("WEEK").alias('N'))
df_group_visit_weekly_2 = df_group_visit_weekly_2.filter(F.col('Developer split_2')=='SPW Group')

# debug
# print(df_group_visit_weekly_.count())
# print(df_group_visit_weekly.count())
# print(df_group_visit_weekly_2.count())

# CPN_CBD
df_group_visit_weekly = df_group_visit_weekly.withColumn('cpn_cbd_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_1')=='CPN_CBD') , F.lit(1))
  .otherwise(F.lit(0))
)

# CPN_Non-CBD
df_group_visit_weekly = df_group_visit_weekly.withColumn('cpn_non_cbd_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_1')=='CPN_Non-CBD') , F.lit(1))
  .otherwise(F.lit(0))
)

# SPW_ICONICS
df_group_visit_weekly = df_group_visit_weekly.withColumn('spw_iconics_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_1')=='SPW_ICONICS') , F.lit(1))
  .otherwise(F.lit(0))
)

# SPW_SPDSCSD
df_group_visit_weekly = df_group_visit_weekly.withColumn('spw_spdscsd_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_1')=='SPW_SPDSCSD') , F.lit(1))
  .otherwise(F.lit(0))
)

# TCC
df_group_visit_weekly = df_group_visit_weekly.withColumn('tcc_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_1')=='TCC') , F.lit(1))
  .otherwise(F.lit(0))
)
# The Mall_CBD
df_group_visit_weekly = df_group_visit_weekly.withColumn('the_mall_cbd_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_1')=='The Mall_CBD'), F.lit(1))
  .otherwise(F.lit(0))
)

# The Mall_Non-CBD
df_group_visit_weekly = df_group_visit_weekly.withColumn('the_mall_non_cbd_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_1')=='The Mall_Non-CBD'), F.lit(1))
  .otherwise(F.lit(0))
)

# SPW Group
df_group_visit_weekly_2 = df_group_visit_weekly_2.withColumn('spw_group_weekly_active', 
  F.when((F.col('N')==n_week) & (F.col('Developer split_2')=='SPW Group'), F.lit(1))
  .otherwise(F.lit(0))
)

# combine every group into one
df_group_visit_weekly = df_group_visit_weekly.groupBy('msisdn').agg(F.sum('cpn_cbd_weekly_active').alias('cpn_cbd_weekly_active'), 
  F.sum('cpn_non_cbd_weekly_active').alias('cpn_non_cbd_weekly_active'), 
  F.sum('spw_iconics_weekly_active').alias('spw_iconics_weekly_active'), 
  F.sum('spw_spdscsd_weekly_active').alias('spw_spdscsd_weekly_active'), 
  F.sum('tcc_weekly_active').alias('tcc_weekly_active'), 
  F.sum('the_mall_cbd_weekly_active').alias('the_mall_cbd_weekly_active'), 
  F.sum('the_mall_non_cbd_weekly_active').alias('the_mall_non_cbd_weekly_active')
)

# join into the main df
df_intermediate_visit_weekly = df_intermediate_visit.join(df_group_visit_weekly, 'msisdn', 'left')
df_group_visit_weekly_2 = df_group_visit_weekly_2.select('msisdn','spw_group_weekly_active')
df_all_columns = df_intermediate_visit_weekly.join(df_group_visit_weekly_2, 'msisdn', 'left')

# debug
# display(df_all_columns.limit(5))

In [0]:
df_all_columns = df_all_columns.fillna(0, subset=[
    'cpn_cbd_customers',
    'cpn_cbd_frequent_customers',
    'cpn_non_cbd_customers',
    'cpn_non_cbd_frequent_customers',
    'spw_iconics_customers',
    'spw_iconics_frequent_customers',
    'spw_spdscsd_customers',
    'spw_spdscsd_frequent_customers',
    'tcc_customers',
    'tcc_frequent_customers',
    'the_mall_cbd_customers',
    'the_mall_cbd_frequent_customers',
    'the_mall_non_cbd_customers',
    'the_mall_non_cbd_frequent_customers',
    'spw_group_customers',
    'spw_group_frequent_customers',
    'cpn_cbd_weekly_active',
    'cpn_non_cbd_weekly_active',
    'spw_iconics_weekly_active',
    'spw_spdscsd_weekly_active',
    'tcc_weekly_active',
    'the_mall_cbd_weekly_active',
    'the_mall_non_cbd_weekly_active',
    'spw_group_weekly_active'])

# debug
# display(df_all_columns.limit(5))

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

In [0]:
col = ['msisdn','name','par_month','cpn_cbd_customers',
    'cpn_cbd_frequent_customers',
    'cpn_non_cbd_customers',
    'cpn_non_cbd_frequent_customers',
    'spw_iconics_customers',
    'spw_iconics_frequent_customers',
    'spw_spdscsd_customers',
    'spw_spdscsd_frequent_customers',
    'tcc_customers',
    'tcc_frequent_customers',
    'the_mall_cbd_customers',
    'the_mall_cbd_frequent_customers',
    'the_mall_non_cbd_customers',
    'the_mall_non_cbd_frequent_customers',
    'spw_group_customers',
    'spw_group_frequent_customers',
    'cpn_cbd_weekly_active',
    'cpn_non_cbd_weekly_active',
    'spw_iconics_weekly_active',
    'spw_spdscsd_weekly_active',
    'tcc_weekly_active',
    'the_mall_cbd_weekly_active',
    'the_mall_non_cbd_weekly_active',
    'spw_group_weekly_active',
    'visit_frequency_(day)', # monthly_by_mall
    'weekly_mall_visitors' # monthly_by_mall
    ,'place_type'
    ,'visit_freq_num'
    ,'latitude'
    ,'longitude'
    ,'province'
    ,'district'
    ,'sub_district'
    ]

In [0]:
save_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/proj_3/{par_month}_flag_freq.parquet'
save_to_parquet(df_all_columns.select(col).distinct(), save_path)

In [0]:
# df = spark.read.format('parquet').load('dbfs:/Volumes/int-cu-siampiwat/staging/raw/202606_flag_freq.parquet')
# df.filter((F.col('visit_freq_num')==4)
#           & (F.col('weekly_mall_visitors')==1)).display()

In [0]:
# df.filter((F.col('msisdn')=='eadeeb3610b78f1e7722cd07adb3edf1127a982ee894ac550ed08cd66da66ab4')).display()

In [0]:
# cols = [
#     'cpn_cbd_customers','cpn_cbd_frequent_customers','cpn_non_cbd_customers','cpn_non_cbd_frequent_customers',
#     'spw_iconics_customers','spw_iconics_frequent_customers','spw_spdscsd_customers','spw_spdscsd_frequent_customers',
#     'tcc_customers','tcc_frequent_customers','the_mall_cbd_customers','the_mall_cbd_frequent_customers',
#     'the_mall_non_cbd_customers','the_mall_non_cbd_frequent_customers','spw_group_customers','spw_group_frequent_customers',
#     'cpn_cbd_weekly_active','cpn_non_cbd_weekly_active','spw_iconics_weekly_active','spw_spdscsd_weekly_active',
#     'tcc_weekly_active','the_mall_cbd_weekly_active','the_mall_non_cbd_weekly_active','spw_group_weekly_active'
# ]

# import functools
# condition = functools.reduce(lambda a, b: a & b, [F.col(c) == 0 for c in cols])

# df_all_zero = df.filter(condition)

# # display result
# display(df_all_zero)

In [0]:
%skip
